In [3]:
# # ROLLING /WINDOW FUNCTIONS
# Rolling functions let you compute statistics over a sliding time window — the foundation of anomaly detection, trend analysis, and dynamic alerting in AIOps.

In [6]:
# rolling() 
import pandas as pd
df_metrics = pd.read_csv("server_metrics.csv")
df_tickets = pd.read_csv("incidents.csv")
df_logs = pd.read_csv("app_logs.csv")

In [16]:
df_metrics_sorted = df_metrics.sort_values(["server_id", "timestamp"])
# 3- period rolling mean per server
df_metrics_sorted["cpu_rolling_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x: x.rolling(window=3).mean())
print(df_metrics_sorted["cpu_rolling_mean"])
print(df_metrics_sorted)

0            NaN
500          NaN
5      60.400000
505    71.200000
10     86.866667
         ...    
479    63.866667
484    50.466667
489    41.266667
494    38.166667
499    46.933333
Name: cpu_rolling_mean, Length: 525, dtype: float64
               timestamp server_id  cpu_pct  memory_pct  response_ms  \
0    2026-01-01 00:00:00    srv-01     49.6        94.6        891.8   
500  2026-01-01 00:00:00    srv-01     49.6        94.6        891.8   
5    2026-01-01 01:00:00    srv-01     82.0        43.6        641.4   
505  2026-01-01 01:00:00    srv-01     82.0        43.6        641.4   
10   2026-01-01 02:00:00    srv-01     96.6        82.7       1130.4   
..                   ...       ...      ...         ...          ...   
479  2026-01-04 23:00:00    srv-05     47.3        40.5         79.3   
484  2026-01-05 00:00:00    srv-05     32.6        31.6        919.6   
489  2026-01-05 01:00:00    srv-05     43.9        47.6        273.8   
494  2026-01-05 02:00:00    srv-05     38

In [20]:
# 3-period rolling std - measures volatility
df_metrics_sorted["cpu_rolling_std"] = df_metrics_sorted.groupby("server_id",observed=True)["cpu_pct"].transform(lambda x: x.rolling(window=3).std())
print(df_metrics_sorted[["server_id", "timestamp", "cpu_pct", "cpu_rolling_mean", "cpu_rolling_std"]].head(5))
# AIOps use — rolling mean smooths noise. Rolling std measures how unstable a server is over time.

    server_id            timestamp  cpu_pct  cpu_rolling_mean  cpu_rolling_std
0      srv-01  2026-01-01 00:00:00     49.6               NaN              NaN
500    srv-01  2026-01-01 00:00:00     49.6               NaN              NaN
5      srv-01  2026-01-01 01:00:00     82.0         60.400000        18.706149
505    srv-01  2026-01-01 01:00:00     82.0         71.200000        18.706149
10     srv-01  2026-01-01 02:00:00     96.6         86.866667         8.429314


In [24]:
# min_periods - handling NaN at start of window
# default rolling -first (window-1) rows are Nan
# min_periods=1 fills from first available row
df_metrics_sorted["cpu_rolling_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x:x.rolling(window=3, min_periods=1).mean())
print(df_metrics_sorted[["server_id", "cpu_pct", "cpu_rolling_mean"]].head(5))
# AIOps use — in production you never want NaN in your alert pipeline. min_periods=1 ensures every row has a value.

    server_id  cpu_pct  cpu_rolling_mean
0      srv-01     49.6         49.600000
500    srv-01     49.6         49.600000
5      srv-01     82.0         60.400000
505    srv-01     82.0         71.200000
10     srv-01     96.6         86.866667


In [45]:
# Rolling window for anomaly detection - Z-score method

# Dynamic Z-score using rolling mean and std
df_metrics_sorted = df_metrics.drop_duplicates(subset = ["server_id", "timestamp"]).sort_values(["server_id", "timestamp"])
df_metrics_sorted["cpu_rolling_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x: x.rolling(window=6, min_periods=1).mean())

df_metrics_sorted["cpu_rolling_std"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x: x.rolling(window=6, min_periods=1).std())

df_metrics_sorted["cpu_zscore"] = (
    (df_metrics_sorted["cpu_pct"] - df_metrics_sorted["cpu_rolling_mean"]) /
df_metrics_sorted["cpu_rolling_std"]
)
print(df_metrics_sorted["cpu_zscore"].dtype)
print(df_metrics_sorted["cpu_zscore"].abs().head())

# Flag anomalies - z-score > 2 is a standard threshold

df_metrics_sorted["is_anomaly"] = df_metrics_sorted["cpu_zscore"].abs().fillna(0) > 1
# df_metrics_sorted["is_anomaly"] = df_metrics_sorted["cpu_zscore"].abs() > 2 (GOT NO ANOMALY )
print(f"Anomalies detected: {df_metrics_sorted["is_anomaly"].sum()}")
print(df_metrics_sorted[df_metrics_sorted["is_anomaly"]][["server_id", "timestamp", "cpu_pct", "cpu_rolling_mean", "cpu_zscore"]].head(5))
#print(df_metrics_sorted.head(5))
# AIOps use — static thresholds (cpu > 80) miss context. Dynamic Z-score catches spikes relative to each server's recent baseline — far more accurate.

float64
0          NaN
5     0.707107
10    0.853592
15    0.058506
20    1.461676
Name: cpu_zscore, dtype: float64
Anomalies detected: 160
   server_id            timestamp  cpu_pct  cpu_rolling_mean  cpu_zscore
20    srv-01  2026-01-01 04:00:00     22.5         65.660000   -1.461676
60    srv-01  2026-01-01 12:00:00     24.1         45.133333   -1.226200
65    srv-01  2026-01-01 13:00:00     78.3         52.550000    1.252685
80    srv-01  2026-01-01 16:00:00     28.1         49.166667   -1.014408
90    srv-01  2026-01-01 18:00:00     81.4         59.350000    1.092008


In [ ]:
# Rule to remember
# Window                     Threshold                  Behaviour
# ------                    ------------              --------------
# Small (3-6)               Use > 1                   More sensitive
# Large (12-24)             Use > 2                  Industry standard
# Production AIOps          window=24, threshold=2   One full day baseline

# For your dataset size (100 rows per server) — window=12, threshold=2 is the sweet spot.
# window=6 produces small z-scores — threshold > 1 works for this dataset
# Production rule: window=24 (full day baseline), threshold=2
# Smaller dataset → lower threshold; larger dataset → threshold=2 safe